# Evaluation — Inference + Gemini JudgeGenerates answers from all trained M1-M7 models, then evaluates with Gemini judge.**Step 1:** Generate answers (GPU needed, no API key needed)**Step 2:** Send to Gemini judge (API key needed, no GPU needed)Set your API key: `export GEMINI_API_KEY="your-key"` or create `~/kd_project/.env`

In [ ]:
# Cell 0: Configimport os, sys, json, random, timeimport torchsys.path.insert(0, os.path.expanduser("~/kd_project"))from shared_utils import (    BASE_DIR, RUNS_DIR, STUDENTS, SEED,    load_jsonl, get_gemini_api_key,)PROMPTS_PATH = os.path.join(BASE_DIR, "clinical_pharm_prompts_10000.jsonl")SPLIT_PATH = os.path.join(BASE_DIR, "clinical_pharm_splits_random_8k_1k_1k_seed42.json")N_EVAL = 200# All experiments to evaluateEXPERIMENTS = {}for name in ["m1_additive_multi_loss", "m2_anti_curriculum", "m3_juggler",             "m4_token_confidence_routing", "m5_section_routed",             "m6_lora_merging", "m7_warmstart_from_e7"]:    path = os.path.join(RUNS_DIR, name)    if os.path.exists(path):        EXPERIMENTS[name] = path        print(f"  ✅ {name}")    else:        print(f"  ⏩ {name} (not found)")# Load test IDsall_prompts = {r["id"]: r["prompt"] for r in load_jsonl(PROMPTS_PATH)}with open(SPLIT_PATH) as f:    splits = json.load(f)rng = random.Random(SEED)test_ids = sorted(splits["test_ids"])rng.shuffle(test_ids)eval_ids = test_ids[:N_EVAL]print(f"\nEvaluating {len(eval_ids)} test samples across {len(EXPERIMENTS)} experiments")

In [ ]:
# Cell 1: Inference — generate answers from all trained modelsfrom transformers import AutoTokenizer, AutoModelForCausalLMfrom peft import PeftModelGEN_KW = dict(max_new_tokens=2000, do_sample=False, temperature=0.0)for exp_name, exp_dir in EXPERIMENTS.items():    out_json = os.path.join(BASE_DIR, f"inference__{exp_name}__{N_EVAL}.json")    if os.path.exists(out_json):        print(f"⏩ {exp_name} already done")        continue    print(f"\n{'='*50} {exp_name} {'='*50}")    results = {"experiment": exp_name, "models": {}, "samples": []}    for eid in eval_ids:        results["samples"].append({"id": eid, "prompt": all_prompts[eid], "outputs": {}})    for sname, scfg in STUDENTS.items():        adapter_path = os.path.join(exp_dir, sname)        if not os.path.exists(adapter_path):            print(f"  ⚠️ {sname} not found, skipping")            continue        print(f"  Loading {sname}...")        tokenizer = AutoTokenizer.from_pretrained(scfg["path"], trust_remote_code=True)        if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token        base_model = AutoModelForCausalLM.from_pretrained(            scfg["path"], torch_dtype=torch.bfloat16,            device_map="auto", trust_remote_code=True)        model = PeftModel.from_pretrained(base_model, adapter_path)        model.eval()        results["models"][sname] = {"adapter": adapter_path}        for sample in results["samples"]:            prompt = sample["prompt"]            if scfg["is_instruct"] and hasattr(tokenizer, "apply_chat_template"):                input_text = tokenizer.apply_chat_template(                    [{"role": "user", "content": prompt}],                    tokenize=False, add_generation_prompt=True)            else:                input_text = prompt            inputs = tokenizer(input_text, return_tensors="pt", truncation=True)            inputs = {k: v.to("cuda") for k, v in inputs.items()}            with torch.no_grad():                out = model.generate(**inputs, **GEN_KW)            decoded = tokenizer.decode(out[0], skip_special_tokens=True)            answer = decoded[len(input_text):].lstrip() if decoded.startswith(input_text) else decoded.strip()            sample["outputs"][sname] = {"answer": answer}        del model, base_model; torch.cuda.empty_cache()        print(f"  ✅ {sname}: {len(results['samples'])} samples")    with open(out_json, "w") as f:        json.dump(results, f, ensure_ascii=False, indent=2)    print(f"✅ Saved {out_json}")print("\n✅ All inference done.")

In [ ]:
# Cell 2: Judge evaluation — sends to Geminifrom google import genaifrom google.genai import typesapi_key = get_gemini_api_key()client = genai.Client(vertexai=False, api_key=api_key)JUDGE_MODEL = "gemini-3-pro-preview"print(f"✅ Judge ready: {JUDGE_MODEL}")JUDGE_PROMPT = """You are an expert clinical pharmacology evaluator.QUESTION:{question}Below are {n_answers} candidate answers ({labels}), in random order. Evaluate each independently.{answers_block}For EACH answer, assign scores:- decision_accuracy: 0 or 5- safety_score: 0-5- clinical_correctness: 0-5- completeness: 0-5- coherence: 0-5- format_compliance: 0-5Return ONLY valid JSON:{{  "A1": {{"decision_accuracy": X, "safety_score": X, "clinical_correctness": X, "completeness": X, "coherence": X, "format_compliance": X}},  ...}}Do NOT reveal hidden reasoning. JSON only."""for exp_name in EXPERIMENTS:    inference_json = os.path.join(BASE_DIR, f"inference__{exp_name}__{N_EVAL}.json")    out_jsonl = os.path.join(BASE_DIR, f"judge__{exp_name}__{N_EVAL}.jsonl")    if not os.path.exists(inference_json):        print(f"⚠️ No inference for {exp_name}")        continue    with open(inference_json) as f:        data = json.load(f)    model_names = sorted(data["models"].keys())    done_ids = set()    if os.path.exists(out_jsonl):        for line in open(out_jsonl):            try:                obj = json.loads(line)                if obj.get("status") == "ok": done_ids.add(obj["id"])            except: pass    remaining = [s for s in data["samples"] if s["id"] not in done_ids]    print(f"\n{'='*40} Judge: {exp_name} (done={len(done_ids)}, remaining={len(remaining)}) {'='*40}")    fout = open(out_jsonl, "a")    for sample in remaining:        sid = sample["id"]        rng_local = random.Random(hash(sid) + SEED)        shuffled = list(model_names); rng_local.shuffle(shuffled)        anon_map = {}; lines = []        for j, mn in enumerate(shuffled):            aid = f"A{j+1}"; anon_map[aid] = mn            ans = sample.get("outputs", {}).get(mn, {}).get("answer", "(no answer)")            lines.append(f"--- {aid} ---\n{ans}\n")        prompt = JUDGE_PROMPT.format(            question=sample["prompt"], n_answers=len(shuffled),            labels=", ".join(anon_map.keys()), answers_block="\n".join(lines))        try:            resp = client.models.generate_content(                model=JUDGE_MODEL, contents=prompt,                config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=12000))            raw = resp.text if hasattr(resp, "text") and resp.text else None            scores = {}            if raw:                start = raw.find("{")                if start >= 0:                    depth = 0                    for i in range(start, len(raw)):                        if raw[i] == "{": depth += 1                        elif raw[i] == "}": depth -= 1                        if depth == 0:                            parsed = json.loads(raw[start:i+1])                            for aid, mn in anon_map.items():                                if aid in parsed: scores[mn] = parsed[aid]                            break            record = {"id": sid, "exp": exp_name, "status": "ok" if scores else "parse_error", "scores": scores}        except Exception as e:            record = {"id": sid, "exp": exp_name, "status": "error", "error": repr(e)}        fout.write(json.dumps(record) + "\n"); fout.flush()        done_ids.add(sid)        if len(done_ids) % 10 == 0:            print(f"  {len(done_ids)}/{len(data['samples'])}")    fout.close()    print(f"✅ {exp_name}: {len(done_ids)} total")print("\n✅ All judge evaluations complete!")

In [ ]:
# Cell 3: Aggregate resultsimport pandas as pdmetric_cols = ["decision_accuracy", "safety_score", "clinical_correctness",               "completeness", "coherence", "format_compliance"]all_tables = []for exp_name in EXPERIMENTS:    jsonl_path = os.path.join(BASE_DIR, f"judge__{exp_name}__{N_EVAL}.jsonl")    if not os.path.exists(jsonl_path): continue    rows = []    for line in open(jsonl_path):        obj = json.loads(line)        if obj.get("status") != "ok": continue        for model_name, scores in obj.get("scores", {}).items():            if isinstance(scores, dict):                rec = {"experiment": exp_name, "model": model_name}                rec.update(scores)                rows.append(rec)    if not rows: continue    df = pd.DataFrame(rows)    for c in metric_cols:        if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")    agg = df.groupby("model")[metric_cols].mean().round(3)    agg["reasoning_mean"] = agg[["clinical_correctness", "completeness", "coherence"]].mean(axis=1).round(3)    print(f"\n{'='*60}")    print(f"  {exp_name}")    print(f"{'='*60}")    display(agg)    all_tables.append((exp_name, agg))# Cross-experiment comparisonif all_tables:    print(f"\n\n{'='*60}")    print("  CROSS-EXPERIMENT COMPARISON (averaged across models)")    print(f"{'='*60}")    summary_rows = []    for exp_name, agg in all_tables:        means = agg.mean()        means["experiment"] = exp_name        summary_rows.append(means)    summary = pd.DataFrame(summary_rows).set_index("experiment")    display(summary.round(3))